In [3]:
# test_mujoco.py
import mujoco



print("MuJoCo version:", mujoco.__version__)

MuJoCo version: 3.6.0


Test if Mujoco is importing the right model and scene

In [13]:
import mujoco
import mujoco.viewer
import numpy as np
import mediapy as media
from matplotlib import pyplot as plt
import modern_robotics as mr

np.set_printoptions(precision=4, suppress=True, linewidth=120)
DEBUG = 1

#Load Mujoco model from XML
# MjModel contains the model description (geometry, joints, actuators, etc)
model = mujoco.MjModel.from_xml_path(r"franka_emika_panda\mjx_scene.xml")

# MjData contains the simulation state (pos, vels, forces, etc)
data = mujoco.MjData(model)

# Set the simulation state to the default config
mujoco.mj_resetData(model,data)

print(f"Number of degrees of freedom (DOF): {model.nv}")
print(f"Number of actuators: {model.nu}")
print(f"Number of bodies: {model.nbody}")

mujoco.mj_forward(model,data)

print(f"Number of qpos: {model.nq}")
print(f"Number of joints: {model.njnt}")
print(f"Number of sites: {model.nsite}")

Number of degrees of freedom (DOF): 9
Number of actuators: 8
Number of bodies: 20
Number of qpos: 9
Number of joints: 9
Number of sites: 1


Test the Video and Renderer

In [5]:
# Start at "home" config

data.qpos = [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]

#Create renderer for visualization
renderer= mujoco.Renderer(model, height=480, width=640)

#Get the camera ID
camera_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_CAMERA,"custom")

#Update renderer with initial state and capture first frame
renderer.update_scene(data, camera=camera_id)
frames = [renderer.render()]

#Simulation parameters
simulation_time = 5.0 #sec
dt = model.opt.timestep
fps = 30
n_steps = int(simulation_time/dt)

print(f"\n Running simulation")

for i in range(n_steps):
    #step the sim forward
    mujoco.mj_step(model,data)

    #update visualization
    renderer.update_scene(data, camera=camera_id)

    #Capture frame (every 5th frame for smooth video)
    if i % int(1/fps/dt) == 0:
        frames.append(renderer.render())

#Display the video
print(f"Simulation complete! Displaying Video")
media.show_video(frames, fps=fps)
renderer.close()


 Running simulation
Simulation complete! Displaying Video


This browser does not support the video tag.


In [7]:
#Reset and set a specific configuration
mujoco.mj_resetData(model,data)

#Get EE site ID
site_id = model.site('gripper').id

#Compute forward kinematics
mujoco.mj_forward(model,data)

ee_pos = data.site_xpos[site_id]
ee_rot = data.site_xmat[site_id].reshape(3,3)


if DEBUG == 1:
    print(f"\tEE position: [{ee_pos[0]:.3f}, {ee_pos[1]:.3f}, {ee_pos[2]:.3f}]")
    print(f"EE Rotation: [{ee_rot}]")




	EE position: [0.088, 0.000, 0.826]
EE Rotation: [[[ 0.7071  0.7071  0.    ]
 [ 0.7071 -0.7071  0.    ]
 [ 0.      0.     -1.    ]]]


Get Forward Kinematics and Screw Axis for PoE

In [8]:
def Get_Screw_axis(n):
    #Get body id for link

    link =f"link{n}"
    body_id = model.body(link).id
    #print(f"\nlink{n} id: {body_id}")

    #obtain world position for Link1
    L = data.xpos[body_id]
    #print(f"Link{n}= {L}")

    omega_local = [0,0,1]

    #Get the world orientation of Link1
    R = data.xmat[body_id].reshape(3,3)
    #print(f"\nRotation:\n {R}")
    #Rotate local frame to world coordinates
    omega_world = R @ omega_local
    #print(f"\n omega world = {omega_world}")
    #Compute Screw Axis
    v = np.cross(omega_world,-L)
    #print(f"Screw_axis{n} = {v}")

    Screw_axis = np.hstack((omega_world,v))
    print(f"Screw axis{n} = {Screw_axis}")

    return Screw_axis

#Get M transform matrix from home position
M = np.eye(4)
M[:3,:3] = ee_rot
M[:3,3] = ee_pos
print(f"M matrix: \n {M}")

#Obtain Screw axes 
#start with joint 1
screw_axis1 = Get_Screw_axis(1)
screw_axis2 = Get_Screw_axis(2)
screw_axis3 = Get_Screw_axis(3)
screw_axis4 = Get_Screw_axis(4)
screw_axis5 = Get_Screw_axis(5)
screw_axis6 = Get_Screw_axis(6)
screw_axis7 = Get_Screw_axis(7)








M matrix: 
 [[ 0.7071  0.7071  0.      0.088 ]
 [ 0.7071 -0.7071  0.      0.    ]
 [ 0.      0.     -1.      0.826 ]
 [ 0.      0.      0.      1.    ]]
Screw axis1 = [0. 0. 1. 0. 0. 0.]
Screw axis2 = [ 0.     1.     0.    -0.333  0.     0.   ]
Screw axis3 = [0. 0. 1. 0. 0. 0.]
Screw axis4 = [ 0.     -1.      0.      0.649   0.     -0.0825]
Screw axis5 = [0. 0. 1. 0. 0. 0.]
Screw axis6 = [ 0.    -1.     0.     1.033  0.    -0.   ]
Screw axis7 = [ 0.     0.    -1.    -0.     0.088  0.   ]


In [9]:

slist = np.column_stack((screw_axis1,screw_axis2,screw_axis3,screw_axis4,screw_axis5,screw_axis6,screw_axis7))

theta_list = np.array((0.5,1,0.8,0.3,0.5,0.8,1))

#Using Modern Robotics Library

T_fk = mr.FKinSpace(M,slist,theta_list)

print(f"\nObtained POE T = \n{T_fk}")

#Use mujoco to check (everything up to the 7th link): 
data.qpos[:7] = theta_list

#Get Mujoco EE pose
mujoco.mj_forward(model,data)

#Get end effector site ID (from xml file)
site_id = model.site("gripper").id

pos_mj = data.site_xpos[site_id].copy()
Rot_mj = data.site_xmat[site_id].reshape(3,3).copy()

T_mj = np.eye(4)

T_mj[:3,:3] = Rot_mj
T_mj[:3,3] = pos_mj

print(f"\nObtained mujoco T_mj = \n{T_mj}")




Obtained POE T = 
[[ 0.5468  0.5871 -0.5969  0.4002]
 [ 0.7864 -0.1155  0.6068  0.3721]
 [ 0.2873 -0.8013 -0.5248  0.6842]
 [ 0.      0.      0.      1.    ]]

Obtained mujoco T_mj = 
[[ 0.5468  0.5871 -0.5969  0.4002]
 [ 0.7864 -0.1155  0.6068  0.3721]
 [ 0.2873 -0.8013 -0.5248  0.6842]
 [ 0.      0.      0.      1.    ]]


This code is Used to Test Implementation of Forward Kinematics Comparing it with Mujoco Forward Kinematics over 6 random iterations

In [ ]:
#Test many configurations
def Pose_error(T_mj,T_fk):
    
    Rot_fk = T_fk[:3,:3] 
    pos_fk = T_fk[:3,3] 

    Rot_mj = T_mj[:3, :3]
    pos_mj = T_mj[:3, 3]
    #Check for error: 

    #Check Translational Error
    pos_error = np.linalg.norm(pos_fk-pos_mj)
    print(f"\nTranslational Error is within: {pos_error}\n")


    #Check Rotational Error if False then Rotation is not similar/ or similar enough. 
    # True - means rotation passed and is close enough
    R_err = Rot_fk.T @ Rot_mj

    cos_theta = (np.trace(R_err) - 1.0)/2.0

    cos_theta = np.clip(cos_theta,-1.0,1.0)

    rot_error = np.arccos(cos_theta)

    print(f"Rotational Error is within: {rot_error}\n")
    
    return pos_error, rot_error

if DEBUG == 1:
    for k in range(6):

        upper_limit = 2.9
        lower_limit = -2.9

        theta = np.random.uniform(low=q_min,high=q_max)
        theta_list = theta

        T_fk = mr.FKinSpace(M,slist,theta_list)

        #Use mujoco to check (everything up to the 7th link): 
        data.qpos[:7] = theta_list
        data.qpos[7:] = 0.0

        #Get Mujoco EE pose
        mujoco.mj_forward(model,data)

        #Get end effector site ID (from xml file)
        site_id = model.site("gripper").id

        pos_mj = data.site_xpos[site_id].copy()
        Rot_mj = data.site_xmat[site_id].reshape(3,3).copy()

        T_mj = np.eye(4)

        T_mj[:3,:3] = Rot_mj
        T_mj[:3,3] = pos_mj
        
        print(f"Mujoco pose \n {T_mj}")
        print(f"\nPOE pose \n {T_fk}")

        pos_error, rot_error = Pose_error(T_mj,T_fk)


Mujoco pose 
 [[-0.5209  0.8536  0.0023  0.0188]
 [ 0.8378  0.5117 -0.1904  0.2994]
 [-0.1637 -0.0972 -0.9817 -0.2476]
 [ 0.      0.      0.      1.    ]]

POE pose 
 [[-0.5209  0.8536  0.0023  0.0188]
 [ 0.8378  0.5117 -0.1904  0.2994]
 [-0.1637 -0.0972 -0.9817 -0.2476]
 [ 0.      0.      0.      1.    ]]

Translational Error is within: 9.973257024868759e-16

Rotational Error is within: 7.300048299977716e-08

Mujoco pose 
 [[ 0.6924 -0.6269  0.3572 -0.2857]
 [-0.015  -0.5075 -0.8615  0.2136]
 [ 0.7214  0.5912 -0.3608  0.7636]
 [ 0.      0.      0.      1.    ]]

POE pose 
 [[ 0.6924 -0.6269  0.3572 -0.2857]
 [-0.015  -0.5075 -0.8615  0.2136]
 [ 0.7214  0.5912 -0.3608  0.7636]
 [ 0.      0.      0.      1.    ]]

Translational Error is within: 2.7194799110210365e-16

Rotational Error is within: 6.322027276634106e-08

Mujoco pose 
 [[-0.4386  0.5562  0.7059  0.5435]
 [-0.898  -0.2412 -0.3679 -0.0701]
 [-0.0344 -0.7953  0.6052  0.5097]
 [ 0.      0.      0.      1.    ]]

POE pose 
 [[-0

Test Trajectories using two transforms to check viability of system (and understand better)

In [ ]:
#Trajectory

#Reset and set a specific config (home position)
site_id = mujoco.mj_resetData(model,data)

#Compute forward kinematics
mujoco.mj_forward(model,data)

#Get endeffector site ID 
site_id = model.site('gripper').id
p0 = data.site_xpos[site_id].copy()
R0 = data.site_xmat[site_id].reshape(3,3).copy()

T0 = np.eye(4)
T1 = np.eye(4)

T0[:3,:3] = R0
T0[:3,3] = p0
print(f"T0 is: \n{T0}")


#fill out T1
p1 = np.array([0.05,0,0])
T1_pos =  p1 + p0

#Rotate about z 15 deg
theta =np.deg2rad(15)

Rz = Rz = np.array([
    [np.cos(theta), -np.sin(theta), 0],
    [np.sin(theta),  np.cos(theta), 0],
    [0,              0,             1]
])

#using local coordinates (EE)
T1_rot = R0 @ Rz

T1[:3,:3] = T1_rot
T1[:3,3] = T1_pos

print(f"T1 is: \n{T1}")

#Small trajectory (for testing purposes)
trajectory = [T0,T1]



Start with the test implementation to get the IK solver

In [ ]:
#Inverting matrices in a better way than taking the inverse
def Mat_inv(T):

    Rot = T[:3,:3]
    p = T[:3,3]

    RotInv = Rot.T
    pinv = -RotInv @ p

    T_inv = np.eye(4)
    T_inv[:3,:3] = RotInv
    T_inv[:3,3] = pinv

    return T_inv

def Get_twist(T):

    se3mat = mr.MatrixLog6(T)

    #Twist
    V = mr.se3ToVec(se3mat)

    if DEBUG:
        print(V)

    return V

#Obtain Pose from q_guess using Mujoco and FK
def Get_pose(q_guess):
    
    #Get current pose from Mujoco
    data.qpos[:7] = q_guess
    data.qpos[7:] = 0.0
    mujoco.mj_forward(model,data)

    site_id = model.site("gripper").id

    p = data.site_xpos[site_id].copy()
    R = data.site_xmat[site_id].reshape(3,3).copy()

    T = np.eye(4)

    T[:3,:3] = R
    T[:3,3] = p

    return T

#obtain Trajectory Error to check IK steps
def Traj_err(T_target,T_current):


    if DEBUG:
        print("T_current =\n", T_current)
        print("T_target =\n", T_target)

    T_curr_inv = Mat_inv(T_current)

    #T_err in body frame
    T_err = T_curr_inv @ T_target

    R_err = T_err[:3,:3]
    p_err = T_err[:3,3]

    p_err_mag = round(np.linalg.norm(p_err),3)
    if DEBUG:
        print(f"Translational error magnitude = {p_err_mag}")
        print(f"Translational error (body) = {p_err}")
        print(f"Rotational error = \n {R_err}")

    return T_err



#Pick desired pose target
T_target = T1.copy()

#Get the current position of the joints
q_guess = data.qpos[:7].copy()

T_current = Get_pose(q_guess)

T_err = Traj_err(T_target,T_current)

#Compute 6-Vector Error twist (using MR library)
V_err = Get_twist(T_err)

#Build Jacobian 
J_pos = np.zeros((3,model.nv))
J_rot = np.zeros((3,model.nv))

mujoco.mj_jacSite(model,data,J_pos,J_rot,site_id)

#Combine Rotational and Position Jacobians to obtain right form use only 7 joints ignore last 2
Jac = np.vstack((J_rot[:,:7],J_pos[:,:7]))

#Obtain pseudo-inverse jacobian
J_pinv = np.linalg.pinv(Jac)

#Obtain change in joints
delta_q = J_pinv @ V_err

alpha = 0.3
delta_q = alpha * delta_q

#Obtain new set of joints
q_new = q_guess + delta_q

#Give to Mujoco and test if position improved
#T_new = Traj_err(q_new)



Implement Newton Raphson based on previous code block (Test Step)

In [66]:
#Inverting matrices in a better way than taking the inverse
def Mat_inv(T):

    Rot = T[:3,:3]
    p = T[:3,3]

    RotInv = Rot.T
    pinv = -RotInv @ p

    T_inv = np.eye(4)
    T_inv[:3,:3] = RotInv
    T_inv[:3,3] = pinv

    return T_inv

def Get_twist(T):

    se3mat = mr.MatrixLog6(T)

    #Twist
    V = mr.se3ToVec(se3mat)

    if DEBUG:
        print(V)

    return V

#Obtain Pose from q_guess using Mujoco and FK
def Get_pose(q_guess):
    
    #Get current pose from Mujoco
    data.qpos[:7] = q_guess
    data.qpos[7:] = 0.0
    mujoco.mj_forward(model,data)

    site_id = model.site("gripper").id

    p = data.site_xpos[site_id].copy()
    R = data.site_xmat[site_id].reshape(3,3).copy()

    T = np.eye(4)

    T[:3,:3] = R
    T[:3,3] = p

    return T

#obtain Trajectory Error to check IK steps
def Traj_err(T_target,T_current):


    if DEBUG:
        print("T_current =\n", T_current)
        print("T_target =\n", T_target)

    T_curr_inv = Mat_inv(T_current)

    #T_err in body frame
    T_err = T_curr_inv @ T_target

    R_err = T_err[:3,:3]
    p_err = T_err[:3,3]

    p_err_mag = round(np.linalg.norm(p_err),3)
    if DEBUG:
        print(f"Translational error magnitude = {p_err_mag}")
        print(f"Translational error (body) = {p_err}")
        print(f"Rotational error = \n {R_err}")

    return T_err

def Get_BodyJac(Jac_space,T_current):
    T_currentBody = Mat_inv(T_current)
    Jac_body = mr.Adjoint(T_currentBody)@Jac_space

    return Jac_body

def IK_step(T_Target,q_guess,alpha=0.3):
    
    T_current = Get_pose(q_guess)
    #Calculate error based on current pose compared to our guess pose
    T_err = Traj_err(T_Target,T_current)
    #Compute 6-Vector Error twist (using MR library)
    V_err = Get_twist(T_err)

    site_id = model.site("gripper").id

    #Build Jacobians and obtain the EE space jacobian from Mujoco
    J_pos = np.zeros((3,model.nv))
    J_rot = np.zeros((3,model.nv))
    mujoco.mj_jacSite(model,data,J_pos,J_rot,site_id)

    #Combine Rotational and Position Jacobians to obtain right form use only 7 joints ignore last 2
    Jac_space = np.vstack((J_rot[:,:7],J_pos[:,:7]))

    #Get body Jacobian as EE position is being calculated from the body frame
    Jac_body = Get_BodyJac(Jac_space,T_current)

    #Obtain pseudo-inverse jacobian
    J_pinv = np.linalg.pinv(Jac_body)

    #Obtain change in joints
    delta_q = J_pinv @ V_err

    #Obtain new set of joints
    q_new = q_guess + alpha*delta_q

    return q_new

#Reset and set a specific config (home position)
mujoco.mj_resetData(model,data)
#Compute forward kinematics
mujoco.mj_forward(model,data)
site_id = model.site('gripper').id

trajectory = [T0,T1]


T_target = T1.copy()
#Get the current position of the joints from Mujoco
q_guess = data.qpos[:7].copy()

for i in range(5):
    
    q_guess= IK_step(T_target,q_guess)
    T_current = Get_pose(q_guess)
    T_err = Traj_err(T_target,T_current)
    V_err = Get_twist(T_err)

    error_norm = np.linalg.norm(V_err)
    print(f"iter {i} - The error norm is: {error_norm}")

    T_current = Get_pose(q_guess)
    

q_final = q_guess

print(q_final)



iter 0 - The error norm is: 0.18692849611032936
iter 1 - The error norm is: 0.1314845197451812
iter 2 - The error norm is: 0.09272192714148365
iter 3 - The error norm is: 0.06556494362614401
iter 4 - The error norm is: 0.04648490086019556
[ 0.0924  0.0515 -0.0029 -0.0138  0.0023  0.0654  0.3096]


In [ ]:
# Start at "home" config

data.qpos = np.append(q_final,[0.0,0.0])

#Create renderer for visualization
renderer= mujoco.Renderer(model, height=480, width=640)

#Get the camera ID
camera_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_CAMERA,"custom")

#Update renderer with initial state and capture first frame
renderer.update_scene(data, camera=camera_id)
frames = [renderer.render()]

#Simulation parameters
simulation_time = 5.0 #sec
dt = model.opt.timestep
fps = 30
n_steps = int(simulation_time/dt)

print(f"\n Running simulation")

for i in range(n_steps):
    #step the sim forward
    mujoco.mj_step(model,data)

    #update visualization
    renderer.update_scene(data, camera=camera_id)

    #Capture frame (every 5th frame for smooth video)
    if i % int(1/fps/dt) == 0:
        frames.append(renderer.render())

#Display the video
print(f"Simulation complete! Displaying Video")
media.show_video(frames, fps=fps)
renderer.close()

Trajectory Generator

In [ ]:
#Trajectory Circle

def Make_pose(R,p):
    T = np.eye(4)

    T[:3,:3] = R
    T[:3,3] = p

    return T

def Rotz(theta):
    return np.array([
        [np.cos(theta), -np.sin(theta), 0],
        [np.sin(theta),  np.cos(theta), 0],
        [0,              0,             1]
    ])

def linear_interpolation(pa,pb,s):
    #Use linear interpolation to find intermediate steps
    val = (1-s)*pa + s*pb
    return val

def Generate_circle_trajectory(T_start, radius=0.08, N=60, z_wave_amp=0.02,tilt_angle=0.15):
    trajectory = []

    R0 = T_start[:3,:3]
    p0 = T_start[:3,3]

    # choose center so current pose starts on the circle
    center = p0 - np.array([radius, 0.0, 0.0])
    beta = np.deg2rad(tilt_angle)

    for s in np.linspace(0, 2*np.pi, N):
        # position path
        p = center + np.array([
            radius * np.cos(s),
            radius * np.sin(s),
            z_wave_amp * np.sin(2*s)
        ])

        # orientation path
        R = Rotz(beta*np.sin(s)) @ R0

        T = np.eye(4)
        T[:3,:3] = R
        T[:3,3] = p

        trajectory.append(T)

    return trajectory




